# Enterprise AI Governance Control Tower: Multi-Agent Model Risk Orchestration with Google ADK

**Author:** Christopher Crilly Pienaah  
**Track:** Agents for Business  
**Course:** 5-Day AI Agents: Intensive Vibe Coding Course with Google × Kaggle (June 2026)  
**Regulatory Context:** OSFI Guideline E-23 — Enterprise-Wide Model Risk Management  

---

> *"Most developers build agents that write text. This system builds multi-agent governance infrastructure that evaluates AI models for enterprise readiness, regulatory compliance, and model risk — before they reach production in regulated industries."*

---

## 1. Executive Summary

Canadian financial institutions face an inflection point. OSFI Guideline E-23 — Enterprise-Wide Model Risk Management — establishes expectations for how federally regulated institutions govern AI and machine learning models. Enforcement ramps up through 2027. Most banks are still operationalizing what this means in practice.

This project builds an **Enterprise AI Governance Control Tower**: a multi-agent system that coordinates five specialized AI governance agents to evaluate whether a model is ready for production deployment under OSFI E-23 expectations.

### What It Does

Given a model and its metrics, the Control Tower:

1. Retrieves applicable regulatory requirements (OSFI Navigator)
2. Assesses compliance gaps in model documentation (OSFI Audit Copilot)
3. Evaluates model risk metrics — drift, fairness, performance (Model Risk Dashboard)
4. Measures hallucination risk for generative AI components (GenAI Reliability Framework)
5. Benchmarks fairness against published standards (CanFraudBench)
6. Applies **deterministic guardrails** — rules no LLM can override
7. Generates a structured executive governance report

### The Core Finding

A fraud detection model achieving **AUC 0.969** — excellent by any accuracy standard — is **BLOCKED** from production by the Control Tower due to fairness violations (AIR 0.59) and significant data drift (PSI 0.25).

This is the CanFraudBench finding made operational: **accuracy alone is insufficient for governance in regulated industries.**

### Live Systems Behind This Project

| Agent | Live System | Status |
|---|---|---|
| Regulatory Intelligence | osfi-navigator-frontend.vercel.app | ✅ Live |
| Compliance Assessment | osfi-audit-copilot-frontend.vercel.app | ✅ Live |
| Model Risk Monitoring | model-risk-dashboard.vercel.app | ✅ Live |
| Reliability Evaluation | genai-reliability-framework.vercel.app | ✅ Live |
| Benchmarking | github.com/CrillyPienaah/canfraudbench | ✅ Published |

This notebook is the **orchestration layer** that unifies all five into a single multi-agent governance pipeline.

## 2. Problem: Accuracy Alone Is Not Governance

### The Trap Every Bank Faces

A data science team trains a fraud detection model. It achieves **AUC 0.969** — among the best in the industry. The model sails through technical review. It gets deployed to production.

Six months later, OSFI examines the model during a supervisory review. They find:

- **Adverse Impact Ratio: 0.59** — the model approves 41% fewer transactions from one demographic group than another, far below the 0.80 threshold used in fairness analysis
- **PSI: 0.25** — the population the model is scoring has drifted significantly from its training distribution
- **No fairness testing documented** — the model card does not address OSFI E-23 Section 6.2

The bank faces remediation requirements, enhanced supervision, and reputational risk.

**This is not hypothetical.** It is the exact scenario demonstrated in CanFraudBench ([github.com/CrillyPienaah/canfraudbench](https://github.com/CrillyPienaah/canfraudbench)), published in June 2026.

### The Governance Gap

The problem is not the model. The problem is the **governance infrastructure** around the model. Banks need:

- Automated fairness evaluation before every deployment
- Ongoing drift monitoring tied to governance thresholds
- Regulatory documentation mapped to specific OSFI sections
- Audit trails that survive examiner scrutiny
- Deterministic rules that no business pressure can override

That is what this Control Tower provides.

### Why Multi-Agent Architecture

A single monolithic agent told to "evaluate this model for OSFI compliance" will hallucinate, miss edge cases, and produce inconsistent outputs. The Day 3 whitepaper from this course demonstrated that specialized agents outperform general-purpose agents on complex, multi-domain tasks.

By decomposing governance evaluation into five specialist agents — each with a narrow, well-defined responsibility — we achieve:

- **Separation of concerns** — each agent owns one governance domain
- **Deterministic guardrails** — rules evaluated in code, not in prompts
- **Auditability** — every agent call is logged with a timestamp
- **Extensibility** — add a new governance requirement by adding a new agent

## 3. Architecture: ADK Supervisor + Five Specialized Agents

```
                    Executive User / Risk Officer
                               │
                               ▼
                  ┌─────────────────────────┐
                  │   Governance Supervisor  │  ← ADK-style orchestrator
                  └────────────┬────────────┘
                               │
      ┌──────────┬─────────────┼─────────────┬──────────┐
      ▼          ▼             ▼             ▼          ▼
  OSFI        Audit        Model         GenAI      CanFraud
 Navigator   Copilot       Risk       Reliability    Bench
  Agent       Agent        Agent        Agent        Agent
      │          │             │             │          │
      └──────────┴─────────────┴─────────────┴──────────┘
                               │
                               ▼
                  ┌─────────────────────────┐
                  │  Deterministic Guardrail │  ← No LLM override
                  │       Engine            │
                  └────────────┬────────────┘
                               │
                               ▼
                  ┌─────────────────────────┐
                  │  Executive Report        │
                  │  Generator              │
                  └─────────────────────────┘
```

### ADK Course Concepts Applied

| Course Concept | Implementation |
|---|---|
| **SequentialAgent** (Day 1) | Five agents called in sequence: Regulatory → Compliance → Risk → Reliability → Benchmark |
| **MCP Tool Schemas** (Day 2) | Each live system exposed as an MCP-style tool with defined input/output schemas |
| **Agent Skills** (Day 3) | Each agent is a specialized skill — narrow trigger, focused execution |
| **Deterministic Guardrails** (Day 4) | Validation rules in Python code, not system prompts — tamper-proof |
| **Spec-Driven Development** (Day 5) | Governance thresholds defined as YAML-mappable specs before any code written |

### Why This Architecture Beats a Chatbot

From the Day 5 whitepaper: *"Hard-coding constraints into a system prompt is brittle, contexts overflow, and agents can be 'convinced' to bypass rules via prompt injection. To build production-grade platforms, external, tamper-proof governance is required."*

Every guardrail in this system is a Python `if` statement. No LLM sees the threshold values. No prompt injection can change the outcome.

## 4. Deterministic Guardrail Engine

This is the foundation of the Control Tower. Rules are defined first. The agents serve the rules — not the other way around.

The thresholds below are enterprise governance values used in this demo, mapped to OSFI E-23 model risk expectations. The 0.80 AIR threshold reflects the widely-used 4/5ths (80%) rule in fairness analysis.

In [ ]:
# Enterprise governance thresholds used for this demo,
# mapped to OSFI E-23 model risk expectations.
# The AIR ≥ 0.80 threshold reflects the 4/5ths fairness rule
# widely used in fairness analysis — not a direct OSFI numeric mandate.

GOVERNANCE_THRESHOLDS = {
    # Fairness — Adverse Impact Ratio (4/5ths rule)
    "air_min": 0.80,
    # Model Drift — Population Stability Index
    "psi_max": 0.20,          # > 0.20 = significant drift
    "psi_warning": 0.10,      # 0.10–0.20 = moderate drift, monitor
    # Reliability — Hallucination risk
    "hallucination_max": 0.10,
    # Performance — Minimum AUC for production models
    "auc_min": 0.70,
    # Calibration — Expected Calibration Error
    "ece_max": 0.05,
    # Equal Opportunity — TPR gap between demographic groups
    "tpr_gap_max": 0.10,
}

def evaluate_guardrails(metrics: dict) -> dict:
    """
    Deterministic guardrail evaluation engine.
    Rules are hard-coded — no LLM can override them.
    Returns APPROVED / CONDITIONAL / BLOCKED status
    with a complete audit trail.
    """
    failures = []
    warnings = []
    passed = []

    # Fairness
    if metrics["air"] < GOVERNANCE_THRESHOLDS["air_min"]:
        failures.append({
            "rule": "FAIRNESS_AIR",
            "severity": "CRITICAL",
            "message": f"AIR {metrics['air']:.3f} below minimum {GOVERNANCE_THRESHOLDS['air_min']}",
            "osfi_reference": "OSFI E-23 Section 6.2: Fairness and Bias",
            "remediation": "Retrain with fairness constraints or apply post-processing calibration"
        })
    else:
        passed.append(f"Fairness AIR: {metrics['air']:.3f} ≥ {GOVERNANCE_THRESHOLDS['air_min']} ✓")

    if metrics["tpr_gap"] > GOVERNANCE_THRESHOLDS["tpr_gap_max"]:
        failures.append({
            "rule": "FAIRNESS_EQUAL_OPPORTUNITY",
            "severity": "HIGH",
            "message": f"TPR gap {metrics['tpr_gap']:.3f} exceeds maximum {GOVERNANCE_THRESHOLDS['tpr_gap_max']}",
            "osfi_reference": "OSFI E-23 Section 6.2: Equal Opportunity",
            "remediation": "Apply equalized odds post-processing"
        })
    else:
        passed.append(f"Equal Opportunity TPR gap: {metrics['tpr_gap']:.3f} ✓")

    # Drift
    if metrics["psi"] > GOVERNANCE_THRESHOLDS["psi_max"]:
        failures.append({
            "rule": "DRIFT_PSI_CRITICAL",
            "severity": "CRITICAL",
            "message": f"PSI {metrics['psi']:.3f} indicates significant distribution shift",
            "osfi_reference": "OSFI E-23 Section 5.1: Model Monitoring",
            "remediation": "Immediate model retraining required"
        })
    elif metrics["psi"] > GOVERNANCE_THRESHOLDS["psi_warning"]:
        warnings.append({
            "rule": "DRIFT_PSI_WARNING",
            "message": f"PSI {metrics['psi']:.3f} moderate drift — monitor closely"
        })
    else:
        passed.append(f"Model Drift PSI: {metrics['psi']:.3f} within threshold ✓")

    # Performance
    if metrics["auc"] < GOVERNANCE_THRESHOLDS["auc_min"]:
        failures.append({
            "rule": "PERFORMANCE_AUC",
            "severity": "HIGH",
            "message": f"AUC {metrics['auc']:.3f} below minimum {GOVERNANCE_THRESHOLDS['auc_min']}",
            "osfi_reference": "OSFI E-23 Section 4.2: Model Performance Standards",
            "remediation": "Improve model or replace with better performing alternative"
        })
    else:
        passed.append(f"Performance AUC: {metrics['auc']:.3f} ✓")

    # Hallucination
    if metrics["hallucination_risk"] > GOVERNANCE_THRESHOLDS["hallucination_max"]:
        failures.append({
            "rule": "RELIABILITY_HALLUCINATION",
            "severity": "CRITICAL",
            "message": f"Hallucination risk {metrics['hallucination_risk']:.3f} exceeds maximum",
            "osfi_reference": "OSFI E-23 Section 7.1: Model Reliability",
            "remediation": "Implement RAG with citation validation before deployment"
        })
    else:
        passed.append(f"Hallucination Risk: {metrics['hallucination_risk']:.3f} ✓")

    # Regulatory readiness
    if not metrics["regulatory_citations_present"]:
        failures.append({
            "rule": "REGULATORY_CITATIONS",
            "severity": "HIGH",
            "message": "No regulatory citations present in model documentation",
            "osfi_reference": "OSFI E-23 Section 3.1: Documentation Requirements",
            "remediation": "Add OSFI E-23 citations to model card"
        })
    else:
        passed.append("Regulatory citations present ✓")

    if not metrics["audit_trail_generated"]:
        failures.append({
            "rule": "AUDIT_TRAIL",
            "severity": "HIGH",
            "message": "No audit trail generated",
            "osfi_reference": "OSFI E-23 Section 8.1: Accountability",
            "remediation": "Generate audit trail before submission"
        })
    else:
        passed.append("Audit trail generated ✓")

    critical = [f for f in failures if f["severity"] == "CRITICAL"]
    status = "BLOCKED" if failures else ("CONDITIONAL" if warnings else "APPROVED")

    return {
        "status": status,
        "summary": {
            "total_rules": len(failures) + len(warnings) + len(passed),
            "passed": len(passed),
            "warnings": len(warnings),
            "failures": len(failures),
            "critical_failures": len(critical),
        },
        "failures": failures,
        "warnings": warnings,
        "passed_rules": passed,
        "audit_required": status == "BLOCKED",
        "llm_override_possible": False,
    }

print("✓ Guardrail engine loaded")
print(f"  Fairness threshold (AIR):      ≥ {GOVERNANCE_THRESHOLDS['air_min']}")
print(f"  Drift threshold (PSI):          < {GOVERNANCE_THRESHOLDS['psi_max']}")
print(f"  Hallucination threshold:        < {GOVERNANCE_THRESHOLDS['hallucination_max']}")
print(f"  Minimum AUC:                    ≥ {GOVERNANCE_THRESHOLDS['auc_min']}")
print(f"  LLM override possible:          False")

## 5. MCP-Style Tool Schemas

Each live system is wrapped as an MCP-style tool the Supervisor Agent can call. In production these connect to real Railway/Vercel endpoints via authenticated HTTP. In this notebook they use structured mock responses that faithfully represent the real system outputs.

This is the Day 2 MCP pattern applied to governance: one standardized interface per system, no bespoke wrappers.

In [ ]:
import asyncio
import json
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional


@dataclass
class GovernanceMetrics:
    """Standardized metrics schema. All five agents contribute to this."""
    model_id: str
    model_name: str
    regulatory_framework: str = "OSFI E-23"
    auc: float = 0.0
    ks_statistic: float = 0.0
    ece: float = 0.0
    air: float = 0.0
    demographic_parity: float = 0.0
    equal_opportunity: float = 0.0
    tpr_gap: float = 0.0
    psi: float = 0.0
    csi: float = 0.0
    hallucination_risk: float = 0.0
    grounding_score: float = 0.0
    citation_coverage: float = 0.0
    regulatory_citations_present: bool = False
    audit_trail_generated: bool = False
    osfi_e23_sections_covered: List[str] = field(default_factory=list)


class OSFINavigatorTool:
    """MCP Tool → osfi-navigator-production.up.railway.app"""
    name = "osfi_navigator"

    @staticmethod
    async def call(model_type: str, use_case: str) -> Dict[str, Any]:
        return {
            "tool": "osfi_navigator",
            "applicable_sections": [
                "Section 3: Governance Framework",
                "Section 4: Model Development and Validation",
                "Section 5: Model Monitoring",
                "Section 6: Fairness and Bias",
                "Section 7: Reliability and Robustness",
                "Section 8: Accountability and Audit",
            ],
            "key_requirements": [
                "Model risk governance framework must be documented",
                "Independent validation required before production",
                "Ongoing monitoring with PSI/CSI metrics",
                "Fairness testing using AIR ≥ 0.80 (4/5ths rule)",
                "Hallucination controls for generative AI components",
                "Complete audit trail for all model decisions",
            ],
            "citations_present": True,
            "enforcement_date": "May 2027",
        }


class AuditCopilotTool:
    """MCP Tool → osfi-audit-copilot-production-f87b.up.railway.app"""
    name = "audit_copilot"

    @staticmethod
    async def call(model_id: str, documentation_summary: str) -> Dict[str, Any]:
        gaps = []
        if "fairness" not in documentation_summary.lower():
            gaps.append("Missing: Fairness testing documentation (OSFI E-23 §6.2)")
        if "monitoring" not in documentation_summary.lower():
            gaps.append("Missing: Ongoing monitoring plan (OSFI E-23 §5.1)")
        if "validation" not in documentation_summary.lower():
            gaps.append("Missing: Independent validation evidence (OSFI E-23 §4.3)")
        return {
            "tool": "audit_copilot",
            "compliance_score": max(0, 100 - len(gaps) * 15),
            "compliance_gaps": gaps,
            "audit_trail_generated": True,
            "regulatory_citations_present": len(gaps) == 0,
        }


class ModelRiskDashboardTool:
    """MCP Tool → model-risk-dashboard-production.up.railway.app"""
    name = "model_risk_dashboard"

    @staticmethod
    async def call(model_id: str, metrics_payload: Dict) -> Dict[str, Any]:
        auc  = metrics_payload.get("auc", 0.0)
        psi  = metrics_payload.get("psi", 0.0)
        air  = metrics_payload.get("air", 0.0)
        tpr  = metrics_payload.get("tpr_gap", 0.0)
        score = 100
        if air < 0.80: score -= 25
        if psi > 0.20: score -= 25
        if auc < 0.70: score -= 25
        if tpr > 0.10: score -= 25
        return {
            "tool": "model_risk_dashboard",
            "performance": {"auc": auc},
            "drift": {"psi": psi},
            "fairness": {"air": air, "tpr_gap": tpr},
            "osfi_e23_compliance_score": score,
            "risk_tier": "HIGH" if score < 50 else "MEDIUM" if score < 75 else "LOW",
        }


class GenAIReliabilityTool:
    """MCP Tool → genai-reliability-framework.vercel.app"""
    name = "genai_reliability"

    @staticmethod
    async def call(model_id: str) -> Dict[str, Any]:
        return {
            "tool": "genai_reliability",
            "hallucination_risk": 0.05,
            "grounding_score": 0.92,
            "citation_coverage": 0.88,
            "reliability_grade": "B",
        }


class CanFraudBenchTool:
    """
    MCP Tool → github.com/CrillyPienaah/canfraudbench

    Key finding: AUC 0.969 can FAIL governance due to AIR 0.59.
    Accuracy alone is insufficient for OSFI E-23 compliance.
    """
    name = "canfraudbench"

    @staticmethod
    async def call(model_id: str, air: float, auc: float) -> Dict[str, Any]:
        return {
            "tool": "canfraudbench",
            "submitted_auc": auc,
            "submitted_air": air,
            "canfraudbench_baseline_auc": 0.969,
            "canfraudbench_baseline_air": 0.59,
            "key_insight": (
                "CanFraudBench demonstrates that AUC 0.969 models can FAIL "
                "governance review due to fairness violations (AIR 0.59 < 0.80). "
                "Accuracy alone is insufficient for OSFI E-23 compliance."
            ),
            "governance_eligible": air >= 0.80,
            "recommendation": (
                "ELIGIBLE for production" if air >= 0.80
                else "INELIGIBLE — fairness remediation required"
            ),
        }


print("✓ Five MCP-style tools loaded")
print("  OSFINavigatorTool      → osfi-navigator-production.up.railway.app")
print("  AuditCopilotTool       → osfi-audit-copilot-production-f87b.up.railway.app")
print("  ModelRiskDashboardTool → model-risk-dashboard-production.up.railway.app")
print("  GenAIReliabilityTool   → genai-reliability-framework.vercel.app")
print("  CanFraudBenchTool      → github.com/CrillyPienaah/canfraudbench")

## Governance Supervisor Agent

The ADK-style Supervisor orchestrates all five tools in sequence, aggregates results into a unified metrics object, applies the deterministic guardrails, and generates the executive report.

In [ ]:
class GovernanceSupervisor:
    """
    ADK-style Supervisor Agent.
    Coordinates five specialized governance agents.
    Applies deterministic guardrails.
    Produces executive governance report.
    """

    def __init__(self):
        self.tools = [
            OSFINavigatorTool(),
            AuditCopilotTool(),
            ModelRiskDashboardTool(),
            GenAIReliabilityTool(),
            CanFraudBenchTool(),
        ]

    async def evaluate(self, model_id, model_name, model_type,
                       use_case, documentation_summary, raw_metrics):
        print(f"\n{'═'*60}")
        print(f"  GOVERNANCE EVALUATION: {model_name}")
        print(f"{'═'*60}")

        # Step 1: Regulatory Intelligence
        regulatory = await OSFINavigatorTool.call(model_type, use_case)
        print(f"  [1/5] Regulatory requirements retrieved")

        # Step 2: Compliance Assessment
        compliance = await AuditCopilotTool.call(model_id, documentation_summary)
        print(f"  [2/5] Compliance score: {compliance['compliance_score']}/100")

        # Step 3: Model Risk Metrics
        risk = await ModelRiskDashboardTool.call(model_id, raw_metrics)
        print(f"  [3/5] Risk tier: {risk['risk_tier']}")

        # Step 4: Reliability Evaluation
        reliability = await GenAIReliabilityTool.call(model_id)
        print(f"  [4/5] Hallucination risk: {reliability['hallucination_risk']}")

        # Step 5: Benchmark
        benchmark = await CanFraudBenchTool.call(
            model_id, raw_metrics.get("air", 0.0), raw_metrics.get("auc", 0.0)
        )
        print(f"  [5/5] Governance eligible: {benchmark['governance_eligible']}")

        # Step 6: Apply deterministic guardrails
        metrics_dict = {
            **raw_metrics,
            "hallucination_risk": reliability["hallucination_risk"],
            "regulatory_citations_present": compliance["regulatory_citations_present"],
            "audit_trail_generated": compliance["audit_trail_generated"],
        }
        guardrails = evaluate_guardrails(metrics_dict)

        status_emoji = {"APPROVED": "✅", "CONDITIONAL": "⚠️", "BLOCKED": "🚫"}[guardrails["status"]]
        print(f"\n  GOVERNANCE STATUS: {status_emoji} {guardrails['status']}")
        print(f"  Rules evaluated:  {guardrails['summary']['total_rules']}")
        print(f"  Passed:           {guardrails['summary']['passed']}")
        print(f"  Failures:         {guardrails['summary']['failures']} "
              f"({guardrails['summary']['critical_failures']} critical)")

        return {
            "model_id": model_id,
            "model_name": model_name,
            "status": guardrails["status"],
            "compliance_score": compliance["compliance_score"],
            "risk_tier": risk["risk_tier"],
            "reliability_grade": reliability["reliability_grade"],
            "guardrails": guardrails,
            "benchmark": benchmark,
            "agent_calls": [
                "OSFINavigatorTool", "AuditCopilotTool",
                "ModelRiskDashboardTool", "GenAIReliabilityTool", "CanFraudBenchTool"
            ],
            "llm_override_possible": False,
        }


print("✓ Governance Supervisor loaded")

## 6. Demo Scenario A: Approved Credit Model

A well-governed credit risk model with proper fairness testing, documentation, and monitoring. Expected outcome: **APPROVED**.

In [ ]:
supervisor = GovernanceSupervisor()

result_a = await supervisor.evaluate(
    model_id="credit_model_v3",
    model_name="TD Credit Risk Scoring Model v3",
    model_type="credit_risk",
    use_case="retail_credit_adjudication",
    documentation_summary=(
        "This model uses XGBoost for credit risk scoring with fairness testing, "
        "independent validation, and ongoing monitoring. All OSFI E-23 sections "
        "have been addressed including fairness, validation, and monitoring."
    ),
    raw_metrics={
        "auc": 0.847,
        "ks": 0.62,
        "ece": 0.018,
        "air": 0.91,          # ✅ Above 0.80
        "tpr_gap": 0.06,      # ✅ Below 0.10
        "psi": 0.08,          # ✅ Below 0.10 warning
        "csi": 0.05,
        "demographic_parity": 0.94,
        "equal_opportunity": 0.89,
    },
)

## 7. Demo Scenario B: High-AUC Fraud Model Blocked

The **CanFraudBench reference scenario**: a fraud detection model achieving AUC 0.969 — excellent performance — that fails governance due to fairness violations and data drift.

This is the core story of this project: **accuracy alone is not governance.**

In [ ]:
result_b = await supervisor.evaluate(
    model_id="fraud_model_v1",
    model_name="CanFraudBench Reference Model — AUC 0.969, AIR 0.59",
    model_type="fraud_detection",
    use_case="transaction_fraud_detection",
    documentation_summary=(
        "High-performance fraud detection model achieving AUC 0.969. "
        "Standard binary classifier trained on transaction data."
        # Deliberately omits: fairness, monitoring, validation
    ),
    raw_metrics={
        "auc": 0.969,         # ✅ Excellent
        "ks": 0.90,
        "ece": 0.001,
        "air": 0.59,          # 🚫 FAILS — below 0.80
        "tpr_gap": 0.18,      # 🚫 FAILS — above 0.10
        "psi": 0.25,          # 🚫 FAILS — above 0.20
        "csi": 0.18,
        "demographic_parity": 0.71,
        "equal_opportunity": 0.68,
    },
)

print("\n  FAILURES DETAIL:")
for f in result_b["guardrails"]["failures"]:
    print(f"  🚫 [{f['severity']}] {f['rule']}")
    print(f"     {f['message']}")
    print(f"     OSFI Ref: {f['osfi_reference']}")
    print(f"     Fix: {f['remediation']}")

## 8. Executive Governance Report

Side-by-side comparison of both scenarios — the format a Chief Risk Officer or OSFI examiner would receive.

In [ ]:
print("\n" + "═"*65)
print("  EXECUTIVE GOVERNANCE REPORT")
print("  Enterprise AI Governance Control Tower")
print("  Generated: " + datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"))
print("═"*65)

print(f"\n  {'Model':<35} {'Status':<12} {'Score':>6} {'Risk':>6}")
print(f"  {'─'*35} {'─'*12} {'─'*6} {'─'*6}")

for result in [result_a, result_b]:
    emoji = {"APPROVED": "✅", "BLOCKED": "🚫", "CONDITIONAL": "⚠️"}[result["status"]]
    name = result["model_name"][:34]
    print(f"  {name:<35} {emoji} {result['status']:<10} "
          f"{result['compliance_score']:>5}/100 {result['risk_tier']:>6}")

print(f"\n  {'─'*65}")
print(f"  KEY FINDING:")
print(f"  The fraud model (AUC 0.969) scores HIGHER on accuracy")
print(f"  but is BLOCKED from production due to:")
print(f"  • Fairness failure:  AIR 0.59  (minimum 0.80)")
print(f"  • Drift failure:     PSI 0.25  (maximum 0.20)")
print(f"  • Documentation gap: Missing fairness, monitoring, validation")
print(f"\n  SOURCE: CanFraudBench — github.com/CrillyPienaah/canfraudbench")
print(f"  REGULATORY: OSFI Guideline E-23 (enforcement May 2027)")
print(f"\n  AUDIT TRAIL: Generated ✓ | LLM Override Possible: False")
print("═"*65)

## 9. Career and Enterprise Impact

### Why This Project Matters for Canadian Banks

OSFI Guideline E-23 establishes expectations for enterprise-wide model risk management at federally regulated financial institutions. The guideline covers:

- Model governance frameworks and accountability
- Development and validation standards
- Ongoing monitoring (PSI, CSI, performance tracking)
- Fairness and bias assessment
- Reliability for generative AI components
- Audit trail and documentation requirements

The Enterprise AI Governance Control Tower addresses every one of these areas through its five specialized agents. It converts abstract regulatory expectations into operational code that a bank's model risk team can run on any model, before every deployment.

### The Blue Ocean Positioning

Most AI engineers build agents that generate text. This project builds governance infrastructure that determines whether AI systems are safe to deploy in regulated industries.

The pitch:

> *"I built a multi-agent AI Governance Control Tower using Google's Agent Development Kit. The system coordinates specialized agents for regulatory intelligence, compliance assessment, model risk monitoring, reliability evaluation, and benchmarking. It helps organizations operationalize AI governance requirements such as OSFI E-23 before models reach production — and it blocks the deployment of models that pass accuracy benchmarks but fail fairness and drift standards."*

### Live Systems Portfolio

| System | URL | Description |
|---|---|---|
| OSFI Navigator | osfi-navigator-frontend.vercel.app | Regulatory intelligence with inline citations |
| OSFI Audit Copilot | osfi-audit-copilot-frontend.vercel.app | Compliance gap assessment |
| Model Risk Dashboard | model-risk-dashboard.vercel.app | PSI drift, AUC, fairness metrics |
| GenAI Reliability Framework | genai-reliability-framework.vercel.app | Hallucination risk evaluation |
| CanFraudBench | github.com/CrillyPienaah/canfraudbench | Published fraud fairness benchmark |
| CanFinBench-SFT | huggingface.co/CrillyPienaah/CanFinBench-SFT-Llama3.2-1B | Fine-tuned financial LLM |

### Course Concepts Demonstrated

| Day | Concept | Implementation |
|---|---|---|
| Day 1 | Multi-agent systems, ADK | GovernanceSupervisor orchestrating 5 agents |
| Day 2 | MCP tools, interoperability | Five MCP-style tools, one per live system |
| Day 3 | Agent skills, progressive disclosure | Each agent loads only when needed |
| Day 4 | Security, deterministic guardrails | 8 hard-coded rules, no LLM override |
| Day 5 | Spec-driven development | Thresholds defined as specs, agents serve the rules |

---

*Christopher Crilly Pienaah | MPS Analytics, Northeastern University (May 2026) | chris-pienaah-portfolio.vercel.app*